In [1]:
from collections import Counter
from pathlib import Path
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sympy import true

import part1 # to use code from tutorial
import torch

# Shared Step: Create Datasets with Three Representations

In [2]:
# splitting data into train and test sets and folds for cross-validation

# merge the party orientations into two classes, "Left" and "Right"
# "Centre" has very few samples, so it is discarded
orientation_to_side = {
    "Centre-left to left": "Left",
    "Centre-left": "Left",
    "Centre to centre-left": "Left",
    "Centre to centre-right": "Right",
    "Centre-right": "Right",
    "Right": "Right",
    "Right to far-right": "Right",
    "Far-right": "Right",
}

def merge_orientations(df, label_column="Party_orientation"):
    df = df[df[label_column].isin(orientation_to_side.keys())].copy()
    df[label_column] = df[label_column].map(orientation_to_side)
    return df

# get the dataframe
df = merge_orientations(pd.read_pickle("./sampled_dataframe.pkl"))

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["Party_orientation"])

# create cross-validation folds
my_seeds = [0] # we can have more
num_folds = 5

# function to create cross-validation folds
# like in tutorial


In [3]:
# to save results
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)
# to save models
my_classifiers = {}

In [4]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in set(x.split()):
            index = self.vocab.get(item)
            if index is not None:
                vector[index] = 1
        return vector

# Function to create pairs of texts and labels so we can use functions from part1 (tutorial)
def make_pairs(df, column, label_names, label_column="Party_orientation"):
    label_to_id = {name: i for i, name in enumerate(label_names)}
    texts = df[column]
    # only join list columns (tokens/lemmas); str.join on a plain string column would put spaces between characters
    if isinstance(texts.iloc[0], (list, tuple, np.ndarray)):
        texts = texts.str.join(" ")
    labels = [label_to_id[name] for name in df[label_column]]
    return list(zip(texts, labels))


In [5]:
df = df.sample(frac=0.2).reset_index(drop=True)

In [6]:
label_names = sorted(df["Party_orientation"].unique())
column = "lemmas"   # "tokens" or "lemmas"
pairs = make_pairs(df, column, label_names)
vocab = get_vocab([text for text, _ in pairs], min_count=5)

In [7]:
my_cross_validation_sets = []
for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(pairs=pairs, k=num_folds, seed=seed)
    )

In [8]:
class BagOfWords(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> np.ndarray:
        vector = np.zeros(len(self))
        for item in x.split():  # no set() here, so repeated words are counted
            index = self.vocab.get(item)
            if index is not None:
                vector[index] += 1
        return vector

In [9]:
class Embedding(Encoder):
    dense = True  # tells part1 to use dense matrices (and lbfgs for logistic regression) instead of sparse ones

    def __init__(self, embeddings_map: pd.DataFrame):
        # build a sentence -> vector dict once, so each lookup is O(1) instead of scanning the whole dataframe
        self.lookup = {
            sentence: np.asarray(embedding, dtype=np.float32)
            for sentence, embedding in zip(embeddings_map['sentence'], embeddings_map['embeddings'])
        }
    def __len__(self) -> int:
        return 1024
    def __call__(self, x: str) -> np.ndarray:
        return self.lookup[x]

In [10]:
embeddingDf = merge_orientations(pd.read_pickle('./embeddings_sampled.pkl'))

In [11]:
# encoders

token_ind_encoder = TokenIndicator(vocab)

bow_encoder = BagOfWords(vocab)

sentence_em_encoder = Embedding(embeddingDf)

#print("vocabulary size:", len(token_ind_encoder))

# Experiment 1: Log-Lin

Exp 1.A: Log-Lin with Token Indicator

In [12]:
my_classifiers["logistic_indicator-lemmas"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_indicator-lemmas/seed-{seed}"
    my_classifiers["logistic_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="logistic",
            save_to=run_folder
        )
    )

accuracies = my_classifiers["logistic_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

fold 0: accuracy 0.6823
fold 1: accuracy 0.6788
fold 2: accuracy 0.6833
fold 3: accuracy 0.6804
fold 4: accuracy 0.6781


Exp 1.B: Log-Lin with Bag of Words

In [13]:
my_classifiers["logistic_indicator-lemmas"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_indicator-lemmas/seed-{seed}"
    my_classifiers["logistic_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="logistic",
            save_to=run_folder
        )
    )

accuracies = my_classifiers["logistic_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

fold 0: accuracy 0.6837
fold 1: accuracy 0.6809
fold 2: accuracy 0.6884
fold 3: accuracy 0.6817
fold 4: accuracy 0.6788


Exp 1.C: Log-Lin with Sentence Embeddings

In [14]:
label_names = sorted(embeddingDf["Party_orientation"].unique())
smallEmbeddingDf = embeddingDf.sample(frac=0.1)
column = "sentence"   # "sentence" for embedding pairs
embedding_pairs = make_pairs(smallEmbeddingDf, column, label_names)

# separate folds for the embedding experiments (my_cross_validation_sets holds lemma strings, not sentences)
embedding_cross_validation_sets = []
for seed in my_seeds:
    embedding_cross_validation_sets.append(
        part1.experiment.cross_validation_split(pairs=embedding_pairs, k=num_folds, seed=seed)
    )

In [15]:
my_classifiers["logistic_embeddings"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_embeddings/seed-{seed}"
    my_classifiers["logistic_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=embedding_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="logistic",
            save_to=run_folder,
            loglin_args= {
                'max_iter': 300
            }
        )
    )

accuracies = my_classifiers["logistic_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

fold 0: accuracy 0.6465
fold 1: accuracy 0.6451
fold 2: accuracy 0.6594
fold 3: accuracy 0.6582
fold 4: accuracy 0.6559


# Experiment 2: Multi-Layer Perceptron
One hidden layer MLP

Exp 2.A: MLP with Token Indicator

In [16]:
my_classifiers["mlp_indicator-lemmas"] = []
outcome_space = part1.clslib.NamedClasses(label_names)

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 0.807 seconds
Dataloader done! Time taken: 11.521 seconds
Training...


  0%|          | 0/20 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/part1/smlp.py:71: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  indices = torch.tensor([x.row, x.col], dtype=torch.long)
/Users/chrisfeltner/Git/nlp/nlp-project/part1/smlp.py:74: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:923.)
  return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()


Epoch 01 | train loss: 0.6238 | train acc: 0.6444 | val loss: 0.5915 | val acc: 0.6708
Epoch 02 | train loss: 0.5584 | train acc: 0.7022 | val loss: 0.5757 | val acc: 0.6869
Epoch 03 | train loss: 0.5225 | train acc: 0.7298 | val loss: 0.5770 | val acc: 0.6889
Epoch 04 | train loss: 0.4967 | train acc: 0.7472 | val loss: 0.5844 | val acc: 0.6868
Epoch 05 | train loss: 0.4750 | train acc: 0.7610 | val loss: 0.5957 | val acc: 0.6869
Epoch 06 | train loss: 0.4562 | train acc: 0.7730 | val loss: 0.6079 | val acc: 0.6871
Epoch 07 | train loss: 0.4379 | train acc: 0.7834 | val loss: 0.6210 | val acc: 0.6843
Epoch 08 | train loss: 0.4201 | train acc: 0.7959 | val loss: 0.6363 | val acc: 0.6827
Epoch 09 | train loss: 0.4016 | train acc: 0.8073 | val loss: 0.6518 | val acc: 0.6820
Epoch 10 | train loss: 0.3830 | train acc: 0.8203 | val loss: 0.6695 | val acc: 0.6786
Epoch 11 | train loss: 0.3640 | train acc: 0.8314 | val loss: 0.6916 | val acc: 0.6802
Epoch 12 | train loss: 0.3438 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

... (3 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6241 | train acc: 0.6449 | val loss: 0.5951 | val acc: 0.6690
Epoch 02 | train loss: 0.5590 | train acc: 0.7007 | val loss: 0.5782 | val acc: 0.6859
Epoch 03 | train loss: 0.5233 | train acc: 0.7285 | val loss: 0.5794 | val acc: 0.6853
Epoch 04 | train loss: 0.4982 | train acc: 0.7471 | val loss: 0.5866 | val acc: 0.6865
Epoch 05 | train loss: 0.4769 | train acc: 0.7609 | val loss: 0.5958 | val acc: 0.6816
Epoch 06 | train loss: 0.4582 | train acc: 0.7716 | val loss: 0.6075 | val acc: 0.6831
Epoch 07 | train loss: 0.4400 | train acc: 0.7834 | val loss: 0.6219 | val acc: 0.6786
Epoch 08 | train loss: 0.4223 | train acc: 0.7954 | val loss: 0.6327 | val acc: 0.6794
Epoch 09 | train loss: 0.4044 | train acc: 0.8063 | val loss: 0.6476 | val acc: 0.6755
Epoch 10 | train loss: 0.3859 | train acc: 0.8191 | val loss: 0.6647 | val acc: 0.6788
Epoch 11 | train loss: 0.3665 | train acc: 0.8307 | val loss: 0.6815 | val acc: 0.6756
Epoch 12 | train loss: 0.3470 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6254 | train acc: 0.6408 | val loss: 0.5937 | val acc: 0.6704
Epoch 02 | train loss: 0.5583 | train acc: 0.7023 | val loss: 0.5799 | val acc: 0.6828
Epoch 03 | train loss: 0.5223 | train acc: 0.7315 | val loss: 0.5825 | val acc: 0.6819
Epoch 04 | train loss: 0.4970 | train acc: 0.7474 | val loss: 0.5905 | val acc: 0.6826
Epoch 05 | train loss: 0.4762 | train acc: 0.7613 | val loss: 0.5978 | val acc: 0.6813
Epoch 06 | train loss: 0.4576 | train acc: 0.7729 | val loss: 0.6105 | val acc: 0.6815
Epoch 07 | train loss: 0.4392 | train acc: 0.7847 | val loss: 0.6218 | val acc: 0.6790
Epoch 08 | train loss: 0.4218 | train acc: 0.7959 | val loss: 0.6338 | val acc: 0.6781
Epoch 09 | train loss: 0.4031 | train acc: 0.8061 | val loss: 0.6508 | val acc: 0.6779
Epoch 10 | train loss: 0.3850 | train acc: 0.8170 | val loss: 0.6702 | val acc: 0.6775
Epoch 11 | train loss: 0.3652 | train acc: 0.8307 | val loss: 0.6896 | val acc: 0.6749
Epoch 12 | train loss: 0.3455 | train acc: 

Exp 2.B: MLP with Bag of Words

In [17]:
my_classifiers["mlp_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_bow-lemmas/seed-{seed}"
    my_classifiers["mlp_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 1.182 seconds
Dataloader done! Time taken: 11.686 seconds
Training...


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6266 | train acc: 0.6420 | val loss: 0.5958 | val acc: 0.6654
Epoch 02 | train loss: 0.5642 | train acc: 0.6978 | val loss: 0.5787 | val acc: 0.6834
Epoch 03 | train loss: 0.5266 | train acc: 0.7285 | val loss: 0.5763 | val acc: 0.6885
Epoch 04 | train loss: 0.4971 | train acc: 0.7495 | val loss: 0.5821 | val acc: 0.6893
Epoch 05 | train loss: 0.4715 | train acc: 0.7671 | val loss: 0.5913 | val acc: 0.6898
Epoch 06 | train loss: 0.4475 | train acc: 0.7818 | val loss: 0.6036 | val acc: 0.6846
Epoch 07 | train loss: 0.4243 | train acc: 0.7962 | val loss: 0.6144 | val acc: 0.6851
Epoch 08 | train loss: 0.4018 | train acc: 0.8100 | val loss: 0.6284 | val acc: 0.6849
Epoch 09 | train loss: 0.3791 | train acc: 0.8253 | val loss: 0.6468 | val acc: 0.6844
Epoch 10 | train loss: 0.3569 | train acc: 0.8385 | val loss: 0.6598 | val acc: 0.6818
Epoch 11 | train loss: 0.3349 | train acc: 0.8533 | val loss: 0.6844 | val acc: 0.6795
Epoch 12 | train loss: 0.3127 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6250 | train acc: 0.6444 | val loss: 0.5967 | val acc: 0.6693
Epoch 02 | train loss: 0.5611 | train acc: 0.7007 | val loss: 0.5812 | val acc: 0.6838
Epoch 03 | train loss: 0.5236 | train acc: 0.7306 | val loss: 0.5816 | val acc: 0.6850
Epoch 04 | train loss: 0.4950 | train acc: 0.7512 | val loss: 0.5883 | val acc: 0.6844
Epoch 05 | train loss: 0.4702 | train acc: 0.7676 | val loss: 0.5966 | val acc: 0.6853
Epoch 06 | train loss: 0.4469 | train acc: 0.7828 | val loss: 0.6087 | val acc: 0.6823
Epoch 07 | train loss: 0.4253 | train acc: 0.7968 | val loss: 0.6265 | val acc: 0.6814
Epoch 08 | train loss: 0.4031 | train acc: 0.8100 | val loss: 0.6353 | val acc: 0.6801
Epoch 09 | train loss: 0.3821 | train acc: 0.8232 | val loss: 0.6523 | val acc: 0.6797
Epoch 10 | train loss: 0.3600 | train acc: 0.8378 | val loss: 0.6724 | val acc: 0.6781
Epoch 11 | train loss: 0.3387 | train acc: 0.8504 | val loss: 0.6911 | val acc: 0.6767
Epoch 12 | train loss: 0.3171 | train acc: 

... (2 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6270 | train acc: 0.6406 | val loss: 0.5987 | val acc: 0.6678
Epoch 02 | train loss: 0.5630 | train acc: 0.6992 | val loss: 0.5818 | val acc: 0.6846
Epoch 03 | train loss: 0.5249 | train acc: 0.7280 | val loss: 0.5796 | val acc: 0.6865
Epoch 04 | train loss: 0.4965 | train acc: 0.7489 | val loss: 0.5853 | val acc: 0.6865
Epoch 05 | train loss: 0.4710 | train acc: 0.7668 | val loss: 0.5935 | val acc: 0.6844
Epoch 06 | train loss: 0.4472 | train acc: 0.7817 | val loss: 0.6043 | val acc: 0.6818
Epoch 07 | train loss: 0.4242 | train acc: 0.7979 | val loss: 0.6191 | val acc: 0.6831
Epoch 08 | train loss: 0.4017 | train acc: 0.8122 | val loss: 0.6342 | val acc: 0.6825
Epoch 09 | train loss: 0.3790 | train acc: 0.8243 | val loss: 0.6474 | val acc: 0.6792
Epoch 10 | train loss: 0.3569 | train acc: 0.8399 | val loss: 0.6681 | val acc: 0.6779
Epoch 11 | train loss: 0.3346 | train acc: 0.8541 | val loss: 0.6864 | val acc: 0.6762
Epoch 12 | train loss: 0.3118 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6273 | train acc: 0.6409 | val loss: 0.5965 | val acc: 0.6683
Epoch 02 | train loss: 0.5630 | train acc: 0.6995 | val loss: 0.5817 | val acc: 0.6806
Epoch 03 | train loss: 0.5262 | train acc: 0.7291 | val loss: 0.5851 | val acc: 0.6825
Epoch 04 | train loss: 0.4983 | train acc: 0.7475 | val loss: 0.5869 | val acc: 0.6822
Epoch 05 | train loss: 0.4741 | train acc: 0.7647 | val loss: 0.5963 | val acc: 0.6801
Epoch 06 | train loss: 0.4513 | train acc: 0.7792 | val loss: 0.6078 | val acc: 0.6801
Epoch 07 | train loss: 0.4301 | train acc: 0.7914 | val loss: 0.6210 | val acc: 0.6801
Epoch 08 | train loss: 0.4088 | train acc: 0.8059 | val loss: 0.6367 | val acc: 0.6786
Epoch 09 | train loss: 0.3883 | train acc: 0.8190 | val loss: 0.6500 | val acc: 0.6771
Epoch 10 | train loss: 0.3668 | train acc: 0.8323 | val loss: 0.6676 | val acc: 0.6773
Epoch 11 | train loss: 0.3457 | train acc: 0.8453 | val loss: 0.6849 | val acc: 0.6756
Epoch 12 | train loss: 0.3246 | train acc: 

Exp 2.C: MLP with Sentence Embeddings

In [18]:
my_classifiers["mlp_embeddings"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_embeddings/seed-{seed}"
    my_classifiers["mlp_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=embedding_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 64,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 20,
            },
        )
    )

accuracies = my_classifiers["mlp_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 0.087 seconds
Dataloader done! Time taken: 0.230 seconds
Training...


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6512 | train acc: 0.6226 | val loss: 0.6401 | val acc: 0.6320
Epoch 02 | train loss: 0.6285 | train acc: 0.6389 | val loss: 0.6317 | val acc: 0.6413
Epoch 03 | train loss: 0.6212 | train acc: 0.6465 | val loss: 0.6285 | val acc: 0.6408
Epoch 04 | train loss: 0.6167 | train acc: 0.6507 | val loss: 0.6257 | val acc: 0.6439
Epoch 05 | train loss: 0.6133 | train acc: 0.6543 | val loss: 0.6239 | val acc: 0.6458
Epoch 06 | train loss: 0.6102 | train acc: 0.6565 | val loss: 0.6225 | val acc: 0.6470
Epoch 07 | train loss: 0.6076 | train acc: 0.6591 | val loss: 0.6214 | val acc: 0.6458
Epoch 08 | train loss: 0.6050 | train acc: 0.6617 | val loss: 0.6201 | val acc: 0.6454
Epoch 09 | train loss: 0.6026 | train acc: 0.6637 | val loss: 0.6205 | val acc: 0.6468
Epoch 10 | train loss: 0.6006 | train acc: 0.6661 | val loss: 0.6194 | val acc: 0.6496
Epoch 11 | train loss: 0.5984 | train acc: 0.6685 | val loss: 0.6181 | val acc: 0.6488
Epoch 12 | train loss: 0.5962 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6563 | train acc: 0.6155 | val loss: 0.6420 | val acc: 0.6281
Epoch 02 | train loss: 0.6311 | train acc: 0.6363 | val loss: 0.6329 | val acc: 0.6364
Epoch 03 | train loss: 0.6229 | train acc: 0.6454 | val loss: 0.6302 | val acc: 0.6386
Epoch 04 | train loss: 0.6183 | train acc: 0.6494 | val loss: 0.6273 | val acc: 0.6403
Epoch 05 | train loss: 0.6150 | train acc: 0.6521 | val loss: 0.6258 | val acc: 0.6417
Epoch 06 | train loss: 0.6122 | train acc: 0.6549 | val loss: 0.6241 | val acc: 0.6420
Epoch 07 | train loss: 0.6095 | train acc: 0.6579 | val loss: 0.6229 | val acc: 0.6435
Epoch 08 | train loss: 0.6067 | train acc: 0.6604 | val loss: 0.6211 | val acc: 0.6470
Epoch 09 | train loss: 0.6047 | train acc: 0.6625 | val loss: 0.6200 | val acc: 0.6477
Epoch 10 | train loss: 0.6025 | train acc: 0.6648 | val loss: 0.6189 | val acc: 0.6486
Epoch 11 | train loss: 0.6002 | train acc: 0.6676 | val loss: 0.6181 | val acc: 0.6496
Epoch 12 | train loss: 0.5976 | train acc: 

... (2 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6532 | train acc: 0.6208 | val loss: 0.6358 | val acc: 0.6315
Epoch 02 | train loss: 0.6310 | train acc: 0.6364 | val loss: 0.6251 | val acc: 0.6423
Epoch 03 | train loss: 0.6239 | train acc: 0.6436 | val loss: 0.6212 | val acc: 0.6434
Epoch 04 | train loss: 0.6196 | train acc: 0.6486 | val loss: 0.6178 | val acc: 0.6494
Epoch 05 | train loss: 0.6163 | train acc: 0.6503 | val loss: 0.6159 | val acc: 0.6513
Epoch 06 | train loss: 0.6135 | train acc: 0.6534 | val loss: 0.6143 | val acc: 0.6540
Epoch 07 | train loss: 0.6107 | train acc: 0.6564 | val loss: 0.6128 | val acc: 0.6560
Epoch 08 | train loss: 0.6084 | train acc: 0.6595 | val loss: 0.6117 | val acc: 0.6570
Epoch 09 | train loss: 0.6061 | train acc: 0.6613 | val loss: 0.6106 | val acc: 0.6595
Epoch 10 | train loss: 0.6040 | train acc: 0.6636 | val loss: 0.6098 | val acc: 0.6589
Epoch 11 | train loss: 0.6021 | train acc: 0.6645 | val loss: 0.6092 | val acc: 0.6578
Epoch 12 | train loss: 0.5997 | train acc: 

  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6571 | train acc: 0.6100 | val loss: 0.6396 | val acc: 0.6275
Epoch 02 | train loss: 0.6328 | train acc: 0.6367 | val loss: 0.6271 | val acc: 0.6413
Epoch 03 | train loss: 0.6251 | train acc: 0.6448 | val loss: 0.6226 | val acc: 0.6452
Epoch 04 | train loss: 0.6205 | train acc: 0.6478 | val loss: 0.6189 | val acc: 0.6448
Epoch 05 | train loss: 0.6170 | train acc: 0.6507 | val loss: 0.6164 | val acc: 0.6496
Epoch 06 | train loss: 0.6141 | train acc: 0.6549 | val loss: 0.6148 | val acc: 0.6514
Epoch 07 | train loss: 0.6115 | train acc: 0.6566 | val loss: 0.6129 | val acc: 0.6514
Epoch 08 | train loss: 0.6092 | train acc: 0.6586 | val loss: 0.6115 | val acc: 0.6535
Epoch 09 | train loss: 0.6070 | train acc: 0.6611 | val loss: 0.6103 | val acc: 0.6541
Epoch 10 | train loss: 0.6047 | train acc: 0.6641 | val loss: 0.6091 | val acc: 0.6573
Epoch 11 | train loss: 0.6028 | train acc: 0.6654 | val loss: 0.6082 | val acc: 0.6591
Epoch 12 | train loss: 0.6001 | train acc: 